# SmartCare Option C - Preprocessing, Feature Engineering and Model Development

This notebook continues the verified dataset-understanding stage. It compares clinical-only and context-augmented feature sets while keeping the final 20% test set untouched until model selection and tuning are complete.

In [ ]:
from pathlib import Path
import json
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from IPython.display import display
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay, RocCurveDisplay, accuracy_score,
    balanced_accuracy_score, classification_report, cohen_kappa_score,
    confusion_matrix, f1_score, mean_absolute_error, precision_score,
    recall_score, roc_auc_score, roc_curve
)
from sklearn.model_selection import (
    RandomizedSearchCV, StratifiedKFold, cross_validate, train_test_split
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, label_binarize
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 100)
sns.set_theme(style="whitegrid", context="notebook")
RANDOM_STATE = 42
CLASS_NAMES = ["Low", "Medium", "High"]
CLASS_TO_INT = {name: index for index, name in enumerate(CLASS_NAMES)}
INT_TO_CLASS = {index: name for name, index in CLASS_TO_INT.items()}

## 1. Load data and establish the prediction target

In [ ]:
PROJECT_DIR = Path.cwd()
DATA_PATH = PROJECT_DIR / "smartcare_ai_dataset_1000.csv"
MODEL_DIR = PROJECT_DIR / "models"
MODEL_DIR.mkdir(exist_ok=True)

df = pd.read_csv(DATA_PATH)
TARGET = "disease_risk_level"

assert set(df[TARGET].unique()) == set(CLASS_NAMES)
assert df[TARGET].isna().sum() == 0

y = df[TARGET].map(CLASS_TO_INT).astype(int)
print(f"Loaded {len(df):,} records and {df.shape[1]} columns")
display(pd.DataFrame({"count": df[TARGET].value_counts().reindex(CLASS_NAMES),
                      "percentage": (df[TARGET].value_counts(normalize=True).reindex(CLASS_NAMES) * 100).round(1)}))

## 3. Create one untouched stratified test set

In [ ]:
train_index, test_index = train_test_split(
    np.arange(len(model_df)),
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

X_train_all = model_df.iloc[train_index].reset_index(drop=True)
X_test_all = model_df.iloc[test_index].reset_index(drop=True)
y_train = y.iloc[train_index].reset_index(drop=True)
y_test = y.iloc[test_index].reset_index(drop=True)

split_check = pd.DataFrame({
    "Full dataset": y.value_counts(normalize=True).sort_index(),
    "Training set": y_train.value_counts(normalize=True).sort_index(),
    "Test set": y_test.value_counts(normalize=True).sort_index(),
}, index=range(3))
split_check.index = CLASS_NAMES
display((split_check * 100).round(1).rename_axis("Risk class (%)"))
print(f"Training rows: {len(y_train)} | Untouched test rows: {len(y_test)}")